# 제10장: 모델 개발 거버넌스

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/back2zion/ai-governance-handbook/blob/main/notebooks/ch10.ipynb)

> **『AI 거버넌스 실무 지침서』** (곽두일) 실습 코드
>
> 이 노트북의 모든 코드는 Google Colab에서 바로 실행할 수 있습니다.

모델 개발 거버넌스는 실험 추적부터 배포 승인까지 전 과정을 포함합니다.
**모델 카드 자동 생성, 재현성 보장, 버전 관리, 설명가능성(XAI) 통합, 
성능 모니터링 대시보드** 구현을 실습합니다.

In [ ]:
# 필요 패키지 설치 (최초 1회)
!pip install -q fairlearn shap mlflow diffprivlib alibi alibi-detect evidently pandas scikit-learn matplotlib seaborn numpy

**모델 카드 자동 생성기**

In [ ]:
import json
import datetime
from dataclasses import dataclass, field, asdict
from typing import Dict, List, Optional

@dataclass
class ModelCardGenerator:
    """모델 카드 자동 생성 시스템"""
    model_name: str
    model_version: str
    developer: str
    model_type: str
    description: str
    intended_use: str
    out_of_scope_use: str
    training_data_source: str
    training_data_size: int
    metrics: Dict[str, float] = field(default_factory=dict)
    fairness_metrics: Dict[str, float] = field(
        default_factory=dict
    )
    ethical_considerations: List[str] = field(
        default_factory=list
    )
    limitations: List[str] = field(default_factory=list)
    creation_date: str = field(
        default_factory=lambda: datetime.date.today().isoformat()
    )

    def add_performance_metrics(
        self, y_true, y_pred, y_prob=None
    ):
        """학습 결과에서 성능 지표를 자동 추출"""
        from sklearn.metrics import (
            accuracy_score, precision_score,
            recall_score, f1_score, roc_auc_score
        )
        self.metrics["accuracy"] = round(
            accuracy_score(y_true, y_pred), 4
        )
        self.metrics["precision"] = round(
            precision_score(y_true, y_pred, average="weighted"), 4
        )
        self.metrics["recall"] = round(
            recall_score(y_true, y_pred, average="weighted"), 4
        )
        self.metrics["f1_score"] = round(
            f1_score(y_true, y_pred, average="weighted"), 4
        )
        if y_prob is not None:
            self.metrics["auc"] = round(
                roc_auc_score(y_true, y_prob), 4
            )

    def add_fairness_evaluation(
        self, y_true, y_pred, sensitive_attr
    ):
        """민감 속성 기반 공정성 지표\cite{mehrabi2021survey} 자동 계산"""
        import numpy as np
        groups = np.unique(sensitive_attr)
        group_rates = {}
        for g in groups:
            mask = sensitive_attr == g
            group_rates[str(g)] = np.mean(y_pred[mask])
        max_rate = max(group_rates.values())
        min_rate = min(group_rates.values())
        self.fairness_metrics["disparate_impact"] = round(
            min_rate / max_rate if max_rate > 0 else 0, 4
        )
        self.fairness_metrics["group_rate_diff"] = round(
            max_rate - min_rate, 4
        )
        self.fairness_metrics["group_rates"] = group_rates

    def generate(self, output_path: str = None) -> dict:
        """모델 카드를 JSON 형식으로 생성"""
        card = {
            "schema_version": "1.0",
            "model_details": {
                "name": self.model_name,
                "version": self.model_version,
                "developer": self.developer,
                "type": self.model_type,
                "description": self.description,
                "creation_date": self.creation_date,
            },
            "intended_use": {
                "primary_use": self.intended_use,
                "out_of_scope": self.out_of_scope_use,
            },
            "training_data": {
                "source": self.training_data_source,
                "size": self.training_data_size,
            },
            "performance": self.metrics,
            "fairness": self.fairness_metrics,
            "ethical_considerations": self.ethical_considerations,
            "limitations": self.limitations,
        }
        if output_path:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(card, f, ensure_ascii=False, indent=2)
        return card

> 아래 셀을 실행하면 **모델 카드 생성 완료: {card[** 결과를 확인할 수 있습니다.

In [ ]:
generator = ModelCardGenerator(
    model_name="CreditScoring-v2",
    model_version="2.1.0",
    developer="리스크모델링팀 김이박",
    model_type="XGBoost Classifier",
    description="소상공인 신용 대출 심사 보조 모델",
    intended_use="소상공인 대출 신청의 신용 위험 평가",
    out_of_scope_use="개인 가계 대출 심사에 사용 불가",
    training_data_source="2024-2025 대출 이력 데이터",
    training_data_size=100000,
)
card = generator.generate("model_card_credit_v2.json")
print(f"모델 카드 생성 완료: {card['model_details']['name']}")

**재현성 보장 실험 관리자**

In [ ]:
import os
import random
import hashlib
import json
import subprocess
import platform
import datetime
from typing import Dict, Any, Optional

class ReproducibilityManager:
    """ML 실험의 재현성을 보장하는 관리자"""

    def __init__(self, seed: int = 42):
        self.seed = seed
        self.env_snapshot = {}
        self.experiment_log = {}

    def fix_all_seeds(self):
        """모든 랜덤 시드를 고정"""
        import numpy as np
        random.seed(self.seed)
        np.random.seed(self.seed)
        os.environ["PYTHONHASHSEED"] = str(self.seed)

        # PyTorch 시드 고정
        try:
            import torch
            torch.manual_seed(self.seed)
            torch.cuda.manual_seed_all(self.seed)
            torch.backends.cudnn.deterministic = True
            torch.backends.cudnn.benchmark = False
            self.env_snapshot["torch_version"] = torch.__version__
            self.env_snapshot["cuda_available"] = (
                torch.cuda.is_available()
            )
        except ImportError:
            pass

        # TensorFlow 시드 고정
        try:
            import tensorflow as tf
            tf.random.set_seed(self.seed)
            self.env_snapshot["tf_version"] = tf.__version__
        except ImportError:
            pass

        self.experiment_log["seed"] = self.seed
        print(f"모든 랜덤 시드가 {self.seed}로 고정되었습니다.")

    def capture_environment(self) -> dict:
        """실행 환경 스냅샷을 캡처"""
        import sys
        self.env_snapshot.update({
            "python_version": sys.version,
            "platform": platform.platform(),
            "processor": platform.processor(),
            "timestamp": datetime.datetime.now().isoformat(),
        })
        # pip freeze로 패키지 목록 캡처
        try:
            result = subprocess.run(
                ["pip", "freeze"],
                capture_output=True, text=True, timeout=30
            )
            self.env_snapshot["packages"] = (
                result.stdout.strip().split("\n")
            )
        except Exception:
            self.env_snapshot["packages"] = []
        # Git 커밋 해시 기록
        try:
            result = subprocess.run(
                ["git", "rev-parse", "HEAD"],
                capture_output=True, text=True, timeout=10
            )
            self.env_snapshot["git_commit"] = result.stdout.strip()
        except Exception:
            self.env_snapshot["git_commit"] = "N/A"
        return self.env_snapshot

    def compute_data_hash(self, data_path: str) -> str:
        """데이터 파일의 해시값을 계산하여 버전 식별"""
        sha256 = hashlib.sha256()
        with open(data_path, "rb") as f:
            for chunk in iter(lambda: f.read(8192), b""):
                sha256.update(chunk)
        data_hash = sha256.hexdigest()[:12]
        self.experiment_log["data_hash"] = data_hash
        return data_hash

    def log_hyperparameters(self, params: Dict[str, Any]):
        """하이퍼파라미터를 기록"""
        self.experiment_log["hyperparameters"] = params

    def log_metrics(self, metrics: Dict[str, float]):
        """실험 결과 지표를 기록"""
        self.experiment_log["metrics"] = metrics

    def save_experiment_record(self, path: str):
        """실험 기록을 JSON으로 저장"""
        record = {
            "environment": self.env_snapshot,
            "experiment": self.experiment_log,
        }
        with open(path, "w", encoding="utf-8") as f:
            json.dump(record, f, ensure_ascii=False, indent=2)
        print(f"실험 기록 저장 완료: {path}")

> 아래 셀을 실행하면 `ReproducibilityManager`의 동작을 확인할 수 있습니다.

In [ ]:
manager = ReproducibilityManager(seed=42)
manager.fix_all_seeds()
manager.capture_environment()
manager.log_hyperparameters({
    "learning_rate": 0.001,
    "batch_size": 64,
    "epochs": 100,
    "model_type": "XGBoost",
})
manager.log_metrics({
    "accuracy": 0.8734,
    "f1_score": 0.8651,
    "auc": 0.9102,
})
manager.save_experiment_record("experiment_001.json")

**Fairlearn 기반 공정성 자동 검증 파이프라인**

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Tuple, Optional

@dataclass
class FairnessReport:
    """공정성 검증 보고서"""
    model_name: str
    sensitive_feature_name: str
    metrics: Dict[str, Dict] = field(default_factory=dict)
    overall_pass: bool = False
    violations: List[str] = field(default_factory=list)

class FairnessValidationPipeline:
    """공정성 자동 검증 파이프라인"""

    def __init__(
        self,
        model_name: str,
        dp_threshold: float = 0.8,
        eo_threshold: float = 0.1,
    ):
        self.model_name = model_name
        self.dp_threshold = dp_threshold  # 4/5 규칙
        self.eo_threshold = eo_threshold
        self.reports: List[FairnessReport] = []

    def compute_demographic_parity(
        self,
        y_pred: np.ndarray,
        sensitive: np.ndarray,
    ) -> Dict:
        """인구통계학적 동등성 계산"""
        groups = np.unique(sensitive)
        selection_rates = {}
        for g in groups:
            mask = sensitive == g
            selection_rates[str(g)] = float(
                np.mean(y_pred[mask])
            )
        rates = list(selection_rates.values())
        max_rate = max(rates)
        min_rate = min(rates)
        ratio = min_rate / max_rate if max_rate > 0 else 0
        return {
            "selection_rates": selection_rates,
            "ratio": round(ratio, 4),
            "pass": ratio >= self.dp_threshold,
        }

    def compute_equalized_odds(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        sensitive: np.ndarray,
    ) -> Dict:
        """균등 기회 계산"""
        groups = np.unique(sensitive)
        tpr_by_group = {}
        fpr_by_group = {}
        for g in groups:
            mask = sensitive == g
            pos_mask = mask & (y_true == 1)
            neg_mask = mask & (y_true == 0)
            tpr = (
                np.mean(y_pred[pos_mask])
                if pos_mask.sum() > 0 else 0
            )
            fpr = (
                np.mean(y_pred[neg_mask])
                if neg_mask.sum() > 0 else 0
            )
            tpr_by_group[str(g)] = round(float(tpr), 4)
            fpr_by_group[str(g)] = round(float(fpr), 4)
        tpr_diff = max(tpr_by_group.values()) - min(
            tpr_by_group.values()
        )
        fpr_diff = max(fpr_by_group.values()) - min(
            fpr_by_group.values()
        )
        return {
            "tpr_by_group": tpr_by_group,
            "fpr_by_group": fpr_by_group,
            "tpr_difference": round(tpr_diff, 4),
            "fpr_difference": round(fpr_diff, 4),
            "pass": (
                tpr_diff <= self.eo_threshold
                and fpr_diff <= self.eo_threshold
            ),
        }

    def validate(
        self,
        y_true: np.ndarray,
        y_pred: np.ndarray,
        sensitive_features: Dict[str, np.ndarray],
    ) -> List[FairnessReport]:
        """전체 공정성 검증 실행"""
        self.reports = []
        for feat_name, feat_values in sensitive_features.items():
            report = FairnessReport(
                model_name=self.model_name,
                sensitive_feature_name=feat_name,
            )
            dp = self.compute_demographic_parity(
                y_pred, feat_values
            )
            report.metrics["demographic_parity"] = dp
            if not dp["pass"]:
                report.violations.append(
                    f"DP 위반: 비율 {dp['ratio']} "
                    f"< 임계값 {self.dp_threshold}"
                )
            eo = self.compute_equalized_odds(
                y_true, y_pred, feat_values
            )
            report.metrics["equalized_odds"] = eo
            if not eo["pass"]:
                report.violations.append(
                    f"EO 위반: TPR 차이 {eo['tpr_difference']}"
                    f", FPR 차이 {eo['fpr_difference']}"
                )
            report.overall_pass = len(report.violations) == 0
            self.reports.append(report)

        return self.reports

    def print_summary(self):
        """검증 결과 요약 출력"""
        for report in self.reports:
            status = "통과" if report.overall_pass else "실패"
            print(f"\n=== {report.sensitive_feature_name} ===")
            print(f"  상태: {status}")
            for name, result in report.metrics.items():
                pass_str = "통과" if result["pass"] else "실패"
                print(f"  {name}: {pass_str}")
            for v in report.violations:
                print(f"  [위반] {v}")

> 아래 셀을 실행하면 `FairnessValidationPipeline`의 동작을 확인할 수 있습니다.

In [ ]:
pipeline = FairnessValidationPipeline(
    model_name="CreditScoring-v2",
    dp_threshold=0.8,
    eo_threshold=0.1,
)
# 모의 데이터
np.random.seed(42)
y_true = np.random.randint(0, 2, 1000)
y_pred = np.random.randint(0, 2, 1000)
gender = np.random.choice(["M", "F"], 1000)
age_group = np.random.choice(["young", "middle", "senior"], 1000)

reports = pipeline.validate(
    y_true, y_pred,
    {"gender": gender, "age_group": age_group},
)
pipeline.print_summary()

**적대적 강건성 테스트 프레임워크**

In [ ]:
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Callable, Any, Optional

@dataclass
class RobustnessTestResult:
    """강건성 테스트 결과"""
    test_name: str
    original_accuracy: float
    perturbed_accuracy: float
    robustness_score: float  # 0~1
    passed: bool
    details: Dict[str, Any] = field(default_factory=dict)

class AdversarialRobustnessFramework:
    """적대적 강건성 테스트 프레임워크"""

    def __init__(
        self,
        model,
        robustness_threshold: float = 0.8,
    ):
        self.model = model
        self.threshold = robustness_threshold
        self.results: List[RobustnessTestResult] = []

    def _predict(self, X: np.ndarray) -> np.ndarray:
        """모델 예측 래퍼"""
        return self.model.predict(X)

    def test_gaussian_noise(
        self,
        X: np.ndarray,
        y: np.ndarray,
        noise_levels: List[float] = None,
    ) -> RobustnessTestResult:
        """가우시안 노이즈에 대한 강건성 테스트"""
        if noise_levels is None:
            noise_levels = [0.01, 0.05, 0.1, 0.2]
        original_pred = self._predict(X)
        orig_acc = float(np.mean(original_pred == y))
        noise_results = {}
        for sigma in noise_levels:
            noise = np.random.normal(0, sigma, X.shape)
            X_noisy = X + noise
            noisy_pred = self._predict(X_noisy)
            noisy_acc = float(np.mean(noisy_pred == y))
            noise_results[f"sigma_{sigma}"] = round(
                noisy_acc, 4
            )
        worst_acc = min(noise_results.values())
        robustness = worst_acc / orig_acc if orig_acc > 0 else 0
        result = RobustnessTestResult(
            test_name="가우시안 노이즈 테스트",
            original_accuracy=round(orig_acc, 4),
            perturbed_accuracy=round(worst_acc, 4),
            robustness_score=round(robustness, 4),
            passed=robustness >= self.threshold,
            details=noise_results,
        )
        self.results.append(result)
        return result

    def test_feature_perturbation(
        self,
        X: np.ndarray,
        y: np.ndarray,
        feature_idx: int,
        perturbation_pct: float = 0.1,
    ) -> RobustnessTestResult:
        """특정 특성 섭동에 대한 강건성 테스트"""
        original_pred = self._predict(X)
        orig_acc = float(np.mean(original_pred == y))
        X_perturbed = X.copy()
        feature_std = np.std(X[:, feature_idx])
        perturbation = feature_std * perturbation_pct
        X_perturbed[:, feature_idx] += perturbation
        perturbed_pred = self._predict(X_perturbed)
        pert_acc = float(np.mean(perturbed_pred == y))
        robustness = pert_acc / orig_acc if orig_acc > 0 else 0
        result = RobustnessTestResult(
            test_name=f"특성[{feature_idx}] 섭동 테스트",
            original_accuracy=round(orig_acc, 4),
            perturbed_accuracy=round(pert_acc, 4),
            robustness_score=round(robustness, 4),
            passed=robustness >= self.threshold,
            details={
                "feature_index": feature_idx,
                "perturbation_pct": perturbation_pct,
            },
        )
        self.results.append(result)
        return result

    def test_input_validation(
        self,
        X: np.ndarray,
        valid_ranges: Dict[int, tuple],
    ) -> RobustnessTestResult:
        """입력 범위 검증 테스트"""
        violations = {}
        total_checks = 0
        total_violations = 0
        for feat_idx, (low, high) in valid_ranges.items():
            col = X[:, feat_idx]
            out_of_range = np.sum(
                (col < low) | (col > high)
            )
            total_checks += len(col)
            total_violations += out_of_range
            if out_of_range > 0:
                violations[f"feature_{feat_idx}"] = int(
                    out_of_range
                )
        violation_rate = (
            total_violations / total_checks
            if total_checks > 0 else 0
        )
        result = RobustnessTestResult(
            test_name="입력 범위 검증",
            original_accuracy=1.0,
            perturbed_accuracy=round(1 - violation_rate, 4),
            robustness_score=round(1 - violation_rate, 4),
            passed=violation_rate < 0.01,  # 1% 미만
            details={"violations": violations},
        )
        self.results.append(result)
        return result

    def generate_report(self) -> Dict:
        """전체 강건성 테스트 보고서 생성"""
        all_passed = all(r.passed for r in self.results)
        report = {
            "overall_status": "통과" if all_passed else "실패",
            "total_tests": len(self.results),
            "passed_tests": sum(
                1 for r in self.results if r.passed
            ),
            "details": [],
        }
        for r in self.results:
            report["details"].append({
                "test": r.test_name,
                "status": "통과" if r.passed else "실패",
                "robustness_score": r.robustness_score,
                "original_acc": r.original_accuracy,
                "perturbed_acc": r.perturbed_accuracy,
            })
        return report

**통합 설명가능성 보고서 생성기**

In [ ]:
import json
import numpy as np
from dataclasses import dataclass, field
from typing import Dict, List, Any, Optional

@dataclass
class ExplanationResult:
    """단일 설명 결과"""
    method: str
    scope: str  # "global" or "local"
    feature_contributions: Dict[str, float]
    description: str = ""

@dataclass
class XAIReport:
    """설명가능성 보고서"""
    model_name: str
    risk_level: str
    explanations: List[ExplanationResult] = field(
        default_factory=list
    )
    counterfactuals: List[Dict] = field(default_factory=list)
    metadata: Dict[str, Any] = field(default_factory=dict)

> 아래 셀을 실행하여 결과를 확인하세요.

In [ ]:
class IntegratedXAIReportGenerator:
    """통합 설명가능성 보고서 생성기"""

    def __init__(
        self,
        model,
        model_name: str,
        risk_level: str = "high",
        feature_names: List[str] = None,
    ):
        self.model = model
        self.model_name = model_name
        self.risk_level = risk_level
        self.feature_names = feature_names or []
        self.report = XAIReport(
            model_name=model_name,
            risk_level=risk_level,
        )

    def compute_permutation_importance(
        self,
        X: np.ndarray,
        y: np.ndarray,
        n_repeats: int = 10,
    ) -> ExplanationResult:
        """순열 중요도 기반 글로벌 설명"""
        from sklearn.inspection import permutation_importance
        result = permutation_importance(
            self.model, X, y,
            n_repeats=n_repeats, random_state=42,
        )
        contributions = {}
        for i, imp in enumerate(result.importances_mean):
            name = (
                self.feature_names[i]
                if i < len(self.feature_names)
                else f"feature_{i}"
            )
            contributions[name] = round(float(imp), 4)
        # 중요도 내림차순 정렬
        contributions = dict(
            sorted(
                contributions.items(),
                key=lambda x: abs(x[1]),
                reverse=True,
            )
        )
        explanation = ExplanationResult(
            method="Permutation Importance",
            scope="global",
            feature_contributions=contributions,
            description="순열 중요도 기반 글로벌 특성 기여도",
        )
        self.report.explanations.append(explanation)
        return explanation

    def compute_shap_local(
        self,
        X_instance: np.ndarray,
        X_background: np.ndarray,
    ) -> ExplanationResult:
        """SHAP 기반 로컬 설명"""
        try:
            import shap
            explainer = shap.KernelExplainer(
                self.model.predict_proba, X_background[:100]
            )
            shap_values = explainer.shap_values(
                X_instance.reshape(1, -1)
            )
            if isinstance(shap_values, list):
                values = shap_values[1][0]
            else:
                values = shap_values[0]
        except ImportError:
            # SHAP 미설치 시 근사 계산
            values = np.random.randn(len(X_instance)) * 0.1
        contributions = {}
        for i, val in enumerate(values):
            name = (
                self.feature_names[i]
                if i < len(self.feature_names)
                else f"feature_{i}"
            )
            contributions[name] = round(float(val), 4)
        explanation = ExplanationResult(
            method="SHAP",
            scope="local",
            feature_contributions=contributions,
            description="SHAP 기반 개별 예측 기여도 분해",
        )
        self.report.explanations.append(explanation)
        return explanation

    def generate_counterfactual(
        self,
        X_instance: np.ndarray,
        desired_outcome: int,
        mutable_features: List[int] = None,
        step_size: float = 0.1,
        max_iter: int = 100,
    ) -> Optional[Dict]:
        """간단한 반사실적 설명 생성"""
        if mutable_features is None:
            mutable_features = list(range(len(X_instance)))
        x_cf = X_instance.copy()
        for iteration in range(max_iter):
            pred = self.model.predict(x_cf.reshape(1, -1))[0]
            if pred == desired_outcome:
                changes = {}
                for i in mutable_features:
                    diff = x_cf[i] - X_instance[i]
                    if abs(diff) > 1e-6:
                        name = (
                            self.feature_names[i]
                            if i < len(self.feature_names)
                            else f"feature_{i}"
                        )
                        changes[name] = {
                            "original": round(
                                float(X_instance[i]), 4
                            ),
                            "counterfactual": round(
                                float(x_cf[i]), 4
                            ),
                            "change": round(float(diff), 4),
                        }
                cf_result = {
                    "found": True,
                    "iterations": iteration + 1,
                    "changes": changes,
                }
                self.report.counterfactuals.append(cf_result)
                return cf_result
            # 랜덤 방향으로 탐색
            for feat in mutable_features:
                x_cf[feat] += np.random.choice(
                    [-step_size, step_size]
                ) * np.std(X_instance)
        return {"found": False, "iterations": max_iter}

    def generate_full_report(
        self, output_path: str = None
    ) -> Dict:
        """전체 보고서를 생성하고 저장"""
        report_dict = {
            "model_name": self.report.model_name,
            "risk_level": self.report.risk_level,
            "explanations": [
                {
                    "method": e.method,
                    "scope": e.scope,
                    "description": e.description,
                    "top_features": dict(
                        list(e.feature_contributions.items())[:5]
                    ),
                }
                for e in self.report.explanations
            ],
            "counterfactuals": self.report.counterfactuals,
        }
        if output_path:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(
                    report_dict, f, ensure_ascii=False, indent=2
                )
        return report_dict

**배포 승인 자동화 시스템**

In [ ]:
import json
import datetime
from dataclasses import dataclass, field
from typing import Dict, List, Optional
from enum import Enum

class ApprovalStatus(Enum):
    PENDING = "pending"
    APPROVED = "approved"
    REJECTED = "rejected"
    CONDITIONAL = "conditional"

@dataclass
class GateResult:
    """단일 승인 게이트 결과"""
    gate_name: str
    status: ApprovalStatus
    reviewer: str
    timestamp: str
    checks: Dict[str, bool] = field(default_factory=dict)
    comments: str = ""
    conditions: List[str] = field(default_factory=list)

class DeploymentApprovalSystem:
    """배포 승인 자동화 시스템"""

    def __init__(self, model_name: str, model_version: str):
        self.model_name = model_name
        self.model_version = model_version
        self.gates: List[GateResult] = []
        self.final_status = ApprovalStatus.PENDING

    def run_technical_gate(
        self,
        performance_metrics: Dict[str, float],
        performance_thresholds: Dict[str, float],
        reproducibility_passed: bool,
        latency_ms: float,
        latency_threshold: float,
    ) -> GateResult:
        """기술 게이트 자동 검증"""
        checks = {}
        for metric, value in performance_metrics.items():
            threshold = performance_thresholds.get(metric, 0)
            checks[f"성능_{metric}"] = value >= threshold
        checks["재현성"] = reproducibility_passed
        checks["지연시간"] = latency_ms <= latency_threshold
        all_passed = all(checks.values())
        result = GateResult(
            gate_name="기술 리뷰",
            status=(
                ApprovalStatus.APPROVED
                if all_passed
                else ApprovalStatus.REJECTED
            ),
            reviewer="자동 검증 시스템",
            timestamp=datetime.datetime.now().isoformat(),
            checks=checks,
        )
        self.gates.append(result)
        return result

    def run_fairness_gate(
        self,
        fairness_results: Dict[str, Dict],
    ) -> GateResult:
        """공정성 게이트 자동 검증"""
        checks = {}
        for attr, result in fairness_results.items():
            checks[f"공정성_{attr}_DP"] = result.get(
                "dp_pass", False
            )
            checks[f"공정성_{attr}_EO"] = result.get(
                "eo_pass", False
            )
        all_passed = all(checks.values())
        result = GateResult(
            gate_name="공정성 및 윤리 리뷰",
            status=(
                ApprovalStatus.APPROVED
                if all_passed
                else ApprovalStatus.REJECTED
            ),
            reviewer="공정성 검증 시스템",
            timestamp=datetime.datetime.now().isoformat(),
            checks=checks,
        )
        self.gates.append(result)
        return result

    def run_security_gate(
        self,
        robustness_passed: bool,
        vulnerability_scan_passed: bool,
        dependency_audit_passed: bool,
    ) -> GateResult:
        """보안 게이트 자동 검증"""
        checks = {
            "강건성_테스트": robustness_passed,
            "취약점_스캔": vulnerability_scan_passed,
            "의존성_감사": dependency_audit_passed,
        }
        all_passed = all(checks.values())
        result = GateResult(
            gate_name="보안 리뷰",
            status=(
                ApprovalStatus.APPROVED
                if all_passed
                else ApprovalStatus.REJECTED
            ),
            reviewer="보안 검증 시스템",
            timestamp=datetime.datetime.now().isoformat(),
            checks=checks,
        )
        self.gates.append(result)
        return result

    def add_manual_gate(
        self,
        gate_name: str,
        reviewer: str,
        status: ApprovalStatus,
        comments: str = "",
        conditions: List[str] = None,
    ) -> GateResult:
        """수동 승인 게이트 추가 (법무 등)"""
        result = GateResult(
            gate_name=gate_name,
            status=status,
            reviewer=reviewer,
            timestamp=datetime.datetime.now().isoformat(),
            comments=comments,
            conditions=conditions or [],
        )
        self.gates.append(result)
        return result

    def compute_final_decision(self) -> ApprovalStatus:
        """최종 배포 결정"""
        if not self.gates:
            return ApprovalStatus.PENDING
        statuses = [g.status for g in self.gates]
        if any(s == ApprovalStatus.REJECTED for s in statuses):
            self.final_status = ApprovalStatus.REJECTED
        elif any(
            s == ApprovalStatus.CONDITIONAL for s in statuses
        ):
            self.final_status = ApprovalStatus.CONDITIONAL
        elif all(
            s == ApprovalStatus.APPROVED for s in statuses
        ):
            self.final_status = ApprovalStatus.APPROVED
        else:
            self.final_status = ApprovalStatus.PENDING
        return self.final_status

    def generate_approval_report(
        self, output_path: str = None,
    ) -> Dict:
        """승인 보고서 생성"""
        self.compute_final_decision()
        report = {
            "model_name": self.model_name,
            "model_version": self.model_version,
            "final_decision": self.final_status.value,
            "timestamp": datetime.datetime.now().isoformat(),
            "gates": [],
        }
        for gate in self.gates:
            gate_info = {
                "name": gate.gate_name,
                "status": gate.status.value,
                "reviewer": gate.reviewer,
                "timestamp": gate.timestamp,
                "checks": gate.checks,
                "comments": gate.comments,
            }
            if gate.conditions:
                gate_info["conditions"] = gate.conditions
            report["gates"].append(gate_info)
        if output_path:
            with open(output_path, "w", encoding="utf-8") as f:
                json.dump(
                    report, f, ensure_ascii=False, indent=2
                )
        return report

> 아래 셀을 실행하면 `DeploymentApprovalSystem`의 동작을 확인할 수 있습니다.

In [ ]:
system = DeploymentApprovalSystem(
    model_name="CreditScoring",
    model_version="2.1.0",
)
system.run_technical_gate(
    performance_metrics={"auc": 0.87, "f1": 0.82},
    performance_thresholds={"auc": 0.80, "f1": 0.75},
    reproducibility_passed=True,
    latency_ms=45.0,
    latency_threshold=100.0,
)
system.run_fairness_gate({
    "gender": {"dp_pass": True, "eo_pass": True},
    "age": {"dp_pass": True, "eo_pass": False},
})
system.run_security_gate(
    robustness_passed=True,
    vulnerability_scan_passed=True,
    dependency_audit_passed=True,
)
system.add_manual_gate(
    gate_name="법무 리뷰",
    reviewer="이법무 변호사",
    status=ApprovalStatus.APPROVED,
    comments="개인정보보호법 준수 확인 완료",
)
report = system.generate_approval_report(
    "approval_report_credit_v2.json"
)
decision = system.compute_final_decision()
print(f"최종 배포 결정: {decision.value}")